In [ ]:
# Data Cleaning as Geometric Preservation
#
# **Key Insight**: Cleaning choices affect the data manifold structure.


import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data_loader import load_raw_data, clean_data

print(" Data Cleaning Analysis")



# Load and clean
df_raw = load_raw_data()
df_clean, meta = clean_data(df_raw)

print(" Cleaning Summary:")
print(f"Original shape: {meta['original_shape']}")
print(f"Cleaned shape: {meta['final_shape']}")
print(f"Removed columns: {meta['removed_columns']}")
print(f"Date range: {meta['date_range'][0]} to {meta['date_range'][1]}")



# Visualize the cleaning impact
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Missing values before/after
ax1 = axes[0, 0]
missing_before = df_raw.isnull().sum()
missing_after = df_clean.isnull().sum()

common_cols = [col for col in df_raw.columns if col in df_clean.columns]
x = range(len(common_cols))

ax1.bar([i-0.2 for i in x], missing_before[common_cols].values, width=0.4, label='Before', alpha=0.7)
ax1.bar([i+0.2 for i in x], missing_after[common_cols].values, width=0.4, label='After', alpha=0.7)
ax1.set_xticks(x)
ax1.set_xticklabels(common_cols, rotation=90, fontsize=8)
ax1.set_ylabel('Missing Values')
ax1.set_title('Missing Values: Before vs After Cleaning')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Distribution comparison for key pollutant
ax2 = axes[0, 1]
if 'CO(GT)' in df_raw.columns and 'CO(GT)' in df_clean.columns:
    ax2.hist(df_raw['CO(GT)'].dropna(), bins=50, alpha=0.5, label='Raw', density=True)
    ax2.hist(df_clean['CO(GT)'], bins=50, alpha=0.5, label='Cleaned', density=True)
    ax2.set_xlabel('CO(GT) Concentration')
    ax2.set_ylabel('Density')
    ax2.set_title('Distribution: Raw vs Cleaned')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

# 3. Time series gap filling
ax3 = axes[1, 0]
if 'CO(GT)' in df_raw.columns:
    sample_range = slice('2004-10-10', '2004-10-20')
    if sample_range.start in df_raw.index and sample_range.stop in df_raw.index:
        df_raw['CO(GT)'].loc[sample_range].plot(ax=ax3, marker='o', linestyle='-',
                                                label='Raw (with gaps)', alpha=0.7)
        df_clean['CO(GT)'].loc[sample_range].plot(ax=ax3, marker='s', linestyle='-',
                                                  label='Cleaned', linewidth=2)
        ax3.set_xlabel('Date')
        ax3.set_ylabel('CO(GT)')
        ax3.set_title('Interpolation Example (10-day sample)')
        ax3.legend()
        ax3.grid(True, alpha=0.3)

# 4. Column removal rationale
ax4 = axes[1, 1]
missing_pct_before = (df_raw.isnull().sum() / len(df_raw)) * 100
columns_removed = meta['removed_columns']
columns_kept = meta['kept_columns']

removed_pct = [missing_pct_before[col] for col in columns_removed]
kept_pct = [missing_pct_before[col] for col in columns_kept if col in missing_pct_before]

ax4.hist(removed_pct, bins=10, alpha=0.7, label=f'Removed ({len(removed_pct)} cols)',
         color='red', density=True)
ax4.hist(kept_pct, bins=10, alpha=0.7, label=f'Kept ({len(kept_pct)} cols)',
         color='green', density=True)
ax4.axvline(x=30, color='black', linestyle='--', label='30% threshold')
ax4.set_xlabel('Missing Percentage')
ax4.set_ylabel('Density')
ax4.set_title('Cleaning Decision: Threshold = 30% missing')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.suptitle('Data Cleaning Analysis: Impact on Data Geometry', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/02_cleaning_impact.png', dpi=150, bbox_inches='tight')
plt.show()



# Save cleaned data
df_clean.to_csv('../data/processed/air_quality_cleaned.csv')
print(" Cleaned data saved to ../data/processed/air_quality_cleaned.csv")



# Geometric implications
print("\n GEOMETRIC IMPLICATIONS OF CLEANING CHOICES:")
print("=" * 60)
print("1. Removing high-missing columns: Reduces dimensionality but may remove")
print("   informative features if missingness is systematic.")
print("\n2. Time-aware interpolation: Preserves temporal manifold structure")
print("   better than simple linear interpolation.")
print("\n3. The 30% threshold: Arbitrary but necessary. Alternative:")
print("   • Keep columns but treat as separate 'reliability' feature")
print("   • Use matrix completion techniques")
print("\nNext: Feature engineering will add structure back meaningfully.")